# Pitch benchmark

Run All refreshes Attune with the production sweep defaults (live and offline: volume 0.06, unvoiced threshold 0.9875). Compatible Attune predictions/checkpoints are reused. Every competitor remains cached; missing competitor rows stop execution. Only the exact Bach10/URMP recordings used in the parameter sweep are excluded; all Coco evaluation stems are retained. Original reports remain immutable. Restart the kernel before Run All.

The expanded evaluation retains 36 Bach10, 136 URMP, and 260 CocoChorales recordings (432 total). Other instrument stems from tuning pieces remain eligible.

Restart the kernel before Run All. Existing compatible per-recording results are reused across cohort changes; only missing or stale Attune recordings are computed. Competitor inference remains disabled. Summary tables and paired tests are rebuilt for the retained tracks. Historical piece-excluded outputs are preserved under `archive/`; cleared outputs must be regenerated before reporting numbers.


### Fair competitor configuration

Every external estimate is now given the same reference/score-derived admissible F0 range. APIs that support range-aware candidate selection (for example Attune, Praat, and Basic Pitch) use it internally; fixed-output APIs are also range-gated after inference. This distinction matters for SwiftF0: its [official API](https://github.com/lars76/swift-f0/) says the network always detects over 46.875–2093.75 Hz and `fmin`/`fmax` only decide which outputs count as voiced. The shared final gate therefore supplies the same information without claiming that it changes SwiftF0's CNN inference. Praat remains raw AC because Praat documents it as the periodicity method; its filtered recommendation is aimed at speech intonation and vocal-fold analysis.

SPICE uses one fixed confidence threshold of **0.35** across every dataset and the streaming adapter. Predictions below it remain unvoiced for scoring. The threshold was chosen after the exploratory confidence sweep; these preliminary results are not a held-out estimate of threshold selection performance.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "app.py").is_file()
    )
)
sys.path.insert(0, str(REPO_ROOT))
import importlib

for module_name in (
    "algorithms.PitchSmoother",
    "benchmarks.modules.pitch.PitchCache",
    "benchmarks.modules.pitch.competitors.Attune",
    "benchmarks.modules.pitch.competitors.Spice",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.datasets.Bach10",
    "benchmarks.modules.pitch.PitchBenchmarker",
    "benchmarks.modules.pitch.PitchNotebook",
):
    importlib.reload(importlib.import_module(module_name))
from benchmarks.modules.pitch.PitchNotebook import PitchNotebook


In [3]:
PRIMARY_METHODS = ("pyin", "attune", "praat", "spice")
RUN_FULL_RUNS = False
CONFIG = PitchNotebook.NotebookConfig(use_cache=True, force_rerun=False, attune_only_refresh=True)
from benchmarks.notebooks.scripts.evaluation_holdout import pitch_notebook, Holdout
BENCH = pitch_notebook(CONFIG)
CACHE_METHODS = tuple(BENCH.methods)
BENCH.show_configuration()
RUN_PAPER_EVALUATION = True
OFFLINE_METHODS = tuple(BENCH.methods)
ONLINE_METHODS = ("attune", "pyin_framewise", "praat", "swiftf0", "spice", "mcleod", "crepe_tiny")
PAPER_ONLINE_METHODS = ONLINE_METHODS
PAIRED_RESAMPLES = 9999


,setting,value
0,workers,8
1,streaming workers,8
2,preliminary tracks per instrument,2
3,full CocoChorales stems per stratum,all
4,random seed,0
5,methods,"pyin, attune, basic_pitch, praat, swiftf0, cre..."
6,estimate caches,reused
7,degradation SNRs,"clean, 20dB, 15dB, 10dB, 5dB"
8,run cache,/Users/sarah/Desktop/shasha/Attune/benchmarks/...


## Method availability

Third-party trackers are optional dependencies, and a missing one is skipped rather than fatal. Run this first so a long cell later does not quietly drop a method: anything marked `MISSING` carries the install hint that would make it available. `torchcrepe` and `penn` are excluded by default because they stack a second heavy PyTorch runtime; set `include_slow_methods=True` on the config to include them, ideally in their own invocation.

In [4]:
availability = BENCH.show_availability()


,method,status,description,detail
0,pyin,OK,Local librosa-compatible pYIN at Attune's 44.1...,
1,attune,OK,Attune pYIN+ promoted production pipeline,
2,basic_pitch,OK,"Basic Pitch contour head, projected to one sub...",
3,praat,OK,Praat autocorrelation (praat-parselmouth),
4,swiftf0,OK,SwiftF0 CNN (swift-f0),
5,crepe,OK,"CREPE CNN, TensorFlow reference implementation",
6,spice,OK,SPICE self-supervised pitch estimator (TF Hub),
7,rmvpe,OK,RMVPE deep U-Net (vendored),
8,mcleod,OK,"Tartini 1.2 native MPM (stateful, official sou...",


## Preliminary sampling check

Every preliminary suite uses the same sampling contract: `CONFIG.preliminary_tracks_per_instrument` tracks for every instrument label, selected independently within each label using `CONFIG.seed`. With the defaults this yields 8 original Bach10 isolated recordings (4 instruments × 2), 26 URMP stems (13 × 2), and 26 CocoChorales stems (13 × 2). The check below also confirms that the same seed repeats exactly while the next seed changes track identities without changing those quotas.

In [5]:
# Frozen exact-track exclusions apply to all returned results.
import pandas as pd
display(pd.DataFrame([{'dataset': d, 'excluded_tuning_tracks': len(p)}
                      for d, p in BENCH.holdout.track_ids.items()]))
print('Filtered report root:', BENCH.holdout.output)


,dataset,excluded_tuning_tracks
0,urmp,13
1,bach10-original,4


Filtered report root: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/evaluation_holdout/604c99165c3537a3


## Preliminary comparison — original Bach10 recordings

Two seed-selected recordings from each of Bach10's four instruments (violin, clarinet, saxophone, bassoon; 8 total). Only isolated monophonic stems are evaluated; ensemble mixes are excluded. Each method's search range is tightened to the reference melody's range.

These are real performances, replacing `bach10-mf0-synth` in this suite. The supplied `*-GTF0s.mat` annotations contain fractional MIDI pitches: the loader converts them to Hz, preserves unvoiced frames, and uses the documented 10 ms hop with the first frame centered at 23 ms. These references were produced with YIN and manual corrections, so they are not independent physical measurements of F0. See the [official description](https://labsites.rochester.edu/air/datasets/Bach10%20Dataset_v1.0.pdf).

Data location: `benchmarks/datasets/bach10-original/<piece>/`. The local v1.1 release comes from this [public mirror, pinned revision](https://github.com/flippy-fyp/Bach10_v1.1/tree/2a53cdc6495bb82f03f943b77ca3b11ddf7f5a31). To provision another checkout, extract that archive into the data location, retaining the ten piece directories. Missing original data raises an error; synthesized audio is never substituted.

This cell runs all configured available methods, then resumes cached results on subsequent runs. Original results use `preliminary_bach10_original`; historical synthesized results under `preliminary` remain unchanged.


In [6]:
preliminary = BENCH.run_preliminary(cache_methods=CACHE_METHODS)


Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[   8/8] attune: 02-AchLiebenChristen-violin  


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9709,0.9715,0.9214,0.9799,0.4644
attune,0.9596,0.9602,0.9287,0.9671,0.3121
basic_pitch,0.9210,0.9223,0.8958,0.9635,0.3005
praat,0.9579,0.9629,0.9312,0.9787,0.2763
swiftf0,0.9251,0.9252,0.8913,0.9370,0.3726
crepe,0.9717,0.9723,0.9302,0.9809,0.3930
spice,0.8972,0.8987,0.8567,0.9320,0.4597
rmvpe,0.9477,0.9506,0.9134,0.9676,0.3537
mcleod,0.9222,0.9247,0.8843,0.9482,0.4116


## Preliminary URMP comparison

Run the same detector set on two seed-selected isolated stems from each of URMP's 13 instrument classes (26 total). The adapter pairs each `AuSep` WAV with its 10 ms `F0s` annotation and carries the piece, ensemble size, instrument, and part number into the result rows. This is the quickest balanced check that the downloaded corpus and every installed method work end to end.

In [7]:
preliminary_urmp = BENCH.run_preliminary_urmp(force=False, cache_methods=CACHE_METHODS)


Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  24/24] attune: 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso      


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9591,0.9654,0.8584,0.9867,0.4740
attune,0.9460,0.9523,0.9339,0.9701,0.1063
basic_pitch,0.9106,0.9134,0.8842,0.9798,0.2031
praat,0.9314,0.9422,0.9275,0.9571,0.0856
swiftf0,0.8508,0.8539,0.8425,0.8708,0.1851
crepe,0.9353,0.9388,0.9249,0.9513,0.1093
spice,0.8921,0.8947,0.8593,0.9470,0.2493
rmvpe,0.9222,0.9412,0.8859,0.9681,0.2337
mcleod,0.9453,0.9576,0.9041,0.9765,0.2318


### Low-latency real-time replay

This live comparison has one controlled pair, one explicitly frame-restarted local pYIN diagnostic, and two neural adapters. Attune's ordinary-candidate frame-local pYIN with the selected causal `0.9875` confidence and `0.06 × running-p95 RMS` gates and raw Praat receive the identical 44.1-kHz capture frame, hop, center timestamp, and score-derived nominal pitch range. `pyin_framewise` runs the local implementation on each newly available 4096-sample, left-aligned frame (`center=False`) at 44.1 kHz. It uses the reference single-frame joint voiced/unvoiced state decision, without an extra 0.5 probability gate or temporal smoothing; it remains distinct from the full-track `pyin` baseline.

SwiftF0 and SPICE use their smallest practical model-native rolling center frames; every update commits one estimate only after its required future samples arrive. SwiftF0 uses 320 ms of receptive context / about 152 ms look-ahead at 16 ms updates, SPICE 64 ms / 32 ms at 32 ms updates. RMVPE is excluded from online runs because its pilot configuration accumulated backlog; it remains in the offline comparison. There is no whole-file call, 1--2 second bounded-history rerun, or timestamp ablation in this table. Model loading and one warm-up call happen before steady-state timing.

The first table uses the notebook's standard pitch-summary columns. The second retains per-recording interaction diagnostics with the same title-case naming convention, and the third shows each adapter's native geometry while making the Attune/Praat match explicit and the frame-restarted librosa geometry visible. Attune uses a `0.06` running-p95 RMS floor in live mode and a `0.06` whole-track p95 RMS floor post hoc. Pass `instruments=()` to sample one stem from every selected corpus instrument, and set `force=True` only when intentionally replacing otherwise current rows.

**Latency retention fix (2026-10-04):** v7 saved only per-recording percentiles, so its exact pooled P95 cannot be recovered. v9 saves every output-update latency, CPU duration, and estimate timestamp in compressed checkpoint sidecars. Ordinary reruns now use a new versioned folder and rerun all methods once; completed v9 recordings are reused only while their latency samples remain valid. Existing v7 results are preserved. Restart the kernel before rerunning after this code update.

Methods use up to all but two logical CPUs (`CONFIG.streaming_workers` sets the ceiling), with concurrency reduced according to available RAM before each method, reusing warmed models within each worker. Progress shows the latest completed track with an elapsed timer. Each completed song/method is checkpointed immediately; rerun with `force=False` to resume, or `force=True` to recompute. Frames remain sequential. Like `pyin_ablation.ipynb`, output latency and deadline misses use dedicated-worker process CPU simulation; measured wall compute time remains a separate diagnostic and can increase under contention. The timing cache version separates these results from the older wall-latency rows.

**pYIN correction (2026-10-04):** v9 removes the nonstandard 50% voiced-probability cutoff. The single-frame decision compares the strongest voiced state with one unvoiced state, matching reference one-frame Viterbi with uniform initialization. Restart the kernel and rerun the online cells; `force=False` uses fresh v9 output folders and preserves older results. Previous streaming pYIN accuracy and paired comparisons must be replaced.


The scheduler leaves two logical CPUs free and reserves the larger of 2 GiB or 15% of physical RAM for the desktop. Each method starts with one calibration job, then budgets measured peak worker memory plus 30% (at least 0.5 GiB per worker). Before a real detection has completed, unmeasured heavy methods retain a 2 GiB budget; cached predictions do not lower it. Offline jobs contain at most four tracks (one for CREPE). CREPE divides the CPU budget across its TensorFlow inference pools: with an eight-core budget, one RAM-limited worker can use eight threads, or two workers can use four threads each. Other methods retain one numerical thread per worker. New CREPE caches record `inference_threads`; older caches remain reusable and retain their original timing. CPU timing still sums process CPU time across threads, so it is not wall throughput. Streaming pools reuse warmed models for batches of eight tracks per worker. Pools release their memory and recheck available RAM between batches. If available RAM falls below the desktop reserve at a batch boundary, execution stops with a resumable error. These are scheduling estimates, not hard memory limits.

**McLeod / Tartini 1.2:** `mcleod` now executes the official C++ MPM engine with its default 2048-sample window and 1024-sample hop at 44.1 kHz. It retains persistent filtering, noise-floor scoring, note tracking, warped NSDF aggregation and retrospective octave correction. Streaming commits causal estimates with 23.22 ms look-ahead; offline reads revised history on the native grid. The common score-range gate applies after inference. See `benchmarks/modules/pitch/competitors/tartini/README.md` for the pinned source, build requirements, explicit fixes, and why the later thesis pipeline differs from the linked release. Timing excludes GUI/device I/O.


In [8]:
streaming_urmp = BENCH.run_streaming_comparison(
    instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)


Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[  22/24] attune: 21/24 finished | 0s | 21 cached; loading model  attune: 3 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  24/24] attune: 23/24 finished | 193s | finished: 14_Waltz_fl_fl_cl/AuSep_1_fl_14_Waltz                 


## Preliminary CocoChorales comparison

The same methods over two seed-selected stems from each of CocoChorales' 13 instrument classes (26 total). The helper prefers two different source compositions within each class and materializes the selected stems automatically. Rows carry ensemble/instrument metadata, and each search range is taken from a robust percentile of the annotated f0 rather than its extremes.

In [9]:
preliminary_coco = BENCH.run_preliminary_coco(force=False, cache_methods=CACHE_METHODS)


materialized 0 file(s) for test
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  26/26] attune: test__woodwind_track228859__2_oboe     


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9648,0.9673,0.9500,0.9868,0.3191
attune,0.9612,0.9636,0.9557,0.9798,0.1434
basic_pitch,0.9658,0.9663,0.9462,0.9959,0.4086
praat,0.9474,0.9575,0.9423,0.9836,0.1491
swiftf0,0.9057,0.9062,0.8954,0.9276,0.2911
crepe,0.9617,0.9633,0.9507,0.9832,0.2489
spice,0.9352,0.9363,0.9215,0.9722,0.3283
rmvpe,0.9260,0.9270,0.9202,0.9680,0.1848
mcleod,0.8479,0.8515,0.8516,0.8710,0.0797


## Offline paired analysis

Pool all five metrics across the eight Bach10, 26 URMP, and 26 CocoChorales preliminary tracks. Compare Attune with each offline competitor on pooled overall accuracy using paired source-piece label swaps and 95% paired cluster-bootstrap intervals. Apply Holm correction within the offline family. Every comparison and the headline table use the same completed track intersection; coverage and excluded tracks are reported. Missing methods raise an error rather than silently changing the test family.

Frames retain equal scoring weight, while dependent frames and stems stay together for inference. `source_groups` can link known shared compositions across corpora. Intervals are unadjusted; adjusted p-values determine significance. These development samples do not establish held-out generalization.


In [10]:
offline_paired = BENCH.show_paired_analysis(
    preliminary,
    preliminary_urmp,
    preliminary_coco,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,8,8,27955,24777,3178
coco,26,26,53986,51162,2824
urmp,24,14,225194,172837,52357


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9614,0.9664,0.8802,0.9860,0.4660
attune,0.9505,0.9554,0.9372,0.9718,0.1193
basic_pitch,0.9230,0.9251,0.8961,0.9815,0.2184
praat,0.9373,0.9474,0.9304,0.9647,0.0990
swiftf0,0.8695,0.8718,0.8562,0.8891,0.2004
crepe,0.9444,0.9472,0.9300,0.9608,0.1315
spice,0.9015,0.9037,0.8700,0.9507,0.2645
rmvpe,0.9255,0.9392,0.8945,0.9680,0.2379
mcleod,0.9230,0.9325,0.8931,0.9520,0.2342


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,58,48,0,307135,0,9999,5.7024,3.0887,8.5608,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,58,48,0,307135,0,9999,4.1112,2.5036,5.9310,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,58,48,0,307135,0,9999,0.6831,-0.0766,1.6453,0.1210,9999,0.1210,False
3,swiftf0,Overall Accuracy,58,48,0,307135,0,9999,8.1030,5.1260,12.9354,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,58,48,0,307135,0,9999,0.7296,0.0834,1.4201,0.0496,9999,0.0992,False
5,spice,Overall Accuracy,58,48,0,307135,0,9999,6.7290,4.4947,9.8699,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,58,48,0,307135,0,9999,4.2792,2.6261,6.2518,0.0001,9999,0.0008,True
7,mcleod,Overall Accuracy,58,48,0,307135,0,9999,4.4143,2.9746,5.9859,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/dd0227efcb57


## Online paired analysis

Add the same eight Bach10 and 26 CocoChorales tracks to the 26-track URMP streaming replay. Pool all five metrics across datasets, then compare overall accuracy, voicing recall, and voicing false-alarm rate against each of the five online competitors. Holm correction covers these fifteen online tests separately from offline tests. Source-piece pairing, confidence intervals, coverage checks, and frame weights follow the offline analysis. The streaming pYIN adaptation is included in all three paired comparisons.


In [11]:
streaming_bach10 = BENCH.run_streaming_comparison(
    dataset="bach10-original", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
streaming_coco = BENCH.run_streaming_comparison(
    dataset="coco", instruments=(), tracks_per_instrument=2, force=False, suite_methods=ONLINE_METHODS
)
online_paired = BENCH.show_paired_analysis(
    streaming_bach10,
    streaming_urmp,
    streaming_coco,
    mode="Streaming",
    methods=ONLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES,
)


Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[   7/8] attune: 6/8 finished | 0s | 6 cached; loading model  attune: 2 worker(s); forced 8-worker policy (limited only by remaining tracks)
[   8/8] attune: 7/8 finished | 27s | finished: 08-FuerDeinenThron-bassoon
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[  26/26] attune: 26/26 finished | 0s | 26 cached


,recordings,output_updates,pooled_p95_output_latency_ms
model,,,
attune,58,1056390,47.0869
pyin_framewise,58,1056390,47.8599
praat,58,1056390,48.2919
swiftf0,58,190837,163.8625
spice,58,95899,36.9421
mcleod,58,132252,23.3820
crepe_tiny,58,33041,63.8768


Streaming: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,8,8,27955,24777,3178
coco,26,26,53986,51162,2824
urmp,24,14,225193,172837,52356


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9475,0.9540,0.9335,0.9725,0.1261
pyin_framewise,0.9639,0.9711,0.8097,0.9993,0.8477
praat,0.9334,0.9543,0.8545,0.9982,0.4815
swiftf0,0.8661,0.8683,0.8555,0.8853,0.1898
spice,0.8538,0.8658,0.8275,0.9140,0.2844
mcleod,0.9206,0.9306,0.8919,0.9518,0.2303
crepe_tiny,0.8658,0.8717,0.8523,0.9237,0.2053


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,58,48,0,307134,0,9999,12.3829,8.7432,15.9168,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,58,48,0,248776,0,9999,-2.6799,-3.8910,-1.6634,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,58,48,0,58358,0,9999,-72.1581,-76.7694,-65.3232,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,58,48,0,307134,0,9999,7.8962,6.3958,9.4479,0.0001,9999,0.0018,True
4,praat,Voicing Recall,58,48,0,248776,0,9999,-2.5714,-3.6972,-1.6026,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,58,48,0,58358,0,9999,-35.5410,-47.5083,-25.7251,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,58,48,0,307134,0,9999,7.7972,4.7987,12.4584,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,58,48,0,248776,0,9999,8.7179,4.2523,15.0219,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,58,48,0,58358,0,9999,-6.3676,-13.6045,-0.4079,0.0632,9999,0.0632,False
9,spice,Overall Accuracy,58,48,0,307134,0,9999,10.5957,7.0814,15.0938,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/6e10633c2ccb


## Expanded filtered paper-cohort evaluation

The offline cell retains the existing evaluation. To run only the updated online experiment, skip the offline cell and use **Expanded online native-geometry experiment** below: first its dry-run cell, then its execution cell. Both require the setup cells only.

The selection remains 40 Bach10, 149 URMP and 260 CocoChorales (20 per instrument), with existing reference alignment, pitch ranges and scoring tolerances. The new comparison uses the same development benchmark set; it is not new held-out validation.


In [12]:
paper_offline = BENCH.run_paper_evaluation("Offline")
paper_offline_paired = BENCH.show_paired_analysis(
    *paper_offline,
    mode="Offline",
    methods=OFFLINE_METHODS,
    n_resamples=PAIRED_RESAMPLES
)

Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  36/36] attune: 10-NunBitten-violin                    


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9730,0.9738,0.9261,0.9841,0.4461
attune,0.9670,0.9678,0.9353,0.9761,0.3168
basic_pitch,0.9254,0.9261,0.8969,0.9699,0.3296
praat,0.9612,0.9631,0.9361,0.9801,0.2634
swiftf0,0.9447,0.9449,0.9060,0.9579,0.4017
crepe,0.9739,0.9746,0.9333,0.9834,0.3900
spice,0.8887,0.8897,0.8497,0.9333,0.4603
rmvpe,0.9626,0.9637,0.9270,0.9819,0.3557
mcleod,0.9416,0.9429,0.9019,0.9639,0.4144


Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/136] attune: 27_King_vn_vn_va_sax/AuSep_1_vn_27_King                Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 128/136] attune: 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 136/136] attune: 44_K515_vn_vn_va_va_vc/AuSep_3_va_44_K515              


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9587,0.9680,0.8701,0.9862,0.4143
attune,0.9463,0.9555,0.9318,0.9706,0.1147
basic_pitch,0.9242,0.9285,0.8969,0.9840,0.1910
praat,0.9288,0.9416,0.9300,0.9543,0.0660
swiftf0,0.8406,0.8450,0.8367,0.8630,0.1756
crepe,0.9431,0.9497,0.9310,0.9624,0.1078
spice,0.9042,0.9094,0.8764,0.9585,0.2128
rmvpe,0.9239,0.9458,0.8916,0.9693,0.2124
mcleod,0.9478,0.9636,0.9090,0.9800,0.2156


materialized 0 file(s) for test
Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[  64/260] attune: test__brass_track223823__4_tuba    Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 128/260] attune: test__random_track235273__1_flute      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 192/260] attune: test__string_track216994__4_cello      Pitch batch (attune): 8/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 256/260] attune: test__woodwind_track229821__3_clarinet Pitch batch (attune): 4/8 worker(s) × 1 inference thread(s); forced Attune policy; no calibration or RAM throttling
[ 260/260] attune: test__woodwind_track229950__4_bassoon  


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9677,0.9705,0.9520,0.9884,0.3288
attune,0.9613,0.9640,0.9550,0.9796,0.1571
basic_pitch,0.9676,0.9685,0.9461,0.9972,0.4390
praat,0.9489,0.9571,0.9429,0.9809,0.1637
swiftf0,0.9156,0.9163,0.9036,0.9379,0.3098
crepe,0.9652,0.9663,0.9524,0.9847,0.2769
spice,0.9367,0.9383,0.9227,0.9757,0.3269
rmvpe,0.9304,0.9319,0.9242,0.9701,0.1873
mcleod,0.8604,0.8639,0.8624,0.8803,0.1019


Offline: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,36,10,118942,105652,13290
coco,260,256,556720,527274,29446
urmp,136,27,1522707,1161066,361641


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
pyin,0.9622,0.9691,0.8939,0.9867,0.4091
attune,0.9519,0.9587,0.9379,0.9736,0.1244
basic_pitch,0.9370,0.9402,0.9093,0.9870,0.2136
praat,0.9366,0.9474,0.9336,0.9637,0.0796
swiftf0,0.8687,0.8718,0.8574,0.8906,0.1928
crepe,0.9514,0.9560,0.9365,0.9702,0.1294
spice,0.9128,0.9168,0.8867,0.9621,0.2292
rmvpe,0.9281,0.9428,0.9017,0.9703,0.2153
mcleod,0.9217,0.9331,0.8968,0.9497,0.2139


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin,Overall Accuracy,432,293,0,2198369,0,9999,4.4012,2.9990,5.5663,0.0001,9999,0.0008,True
1,basic_pitch,Overall Accuracy,432,293,0,2198369,0,9999,2.8538,2.1824,3.6230,0.0001,9999,0.0008,True
2,praat,Overall Accuracy,432,293,0,2198369,0,9999,0.4261,0.0229,0.9257,0.0404,9999,0.0808,False
3,swiftf0,Overall Accuracy,432,293,0,2198369,0,9999,8.0457,6.5600,9.3999,0.0001,9999,0.0008,True
4,crepe,Overall Accuracy,432,293,0,2198369,0,9999,0.1327,-0.1237,0.4103,0.3679,9999,0.3679,False
5,spice,Overall Accuracy,432,293,0,2198369,0,9999,5.1175,4.3577,5.8984,0.0001,9999,0.0008,True
6,rmvpe,Overall Accuracy,432,293,0,2198369,0,9999,3.6140,2.5634,4.5299,0.0001,9999,0.0008,True
7,mcleod,Overall Accuracy,432,293,0,2198369,0,9999,4.1072,3.1379,5.2799,0.0001,9999,0.0008,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/offline/843dc9a1077c


### Expanded online native-geometry experiment (November 2026)

Run the setup cells, then the dry-run and execution cells below for **432 retained tracks (36 Bach10, 136 URMP, 260 CocoChorales)** from the frozen 449-track source cohort. This explicit selective policy supersedes `attune_only_refresh` for this experiment. Keep `force_rerun=False`. Historical online and all offline results remain separate.

Only pYIN (4096/1024 samples at 44.1 kHz; joint frame decision without sequence HMM), Praat (native one-frame raw AC, 3/floor window and 0.75/floor hop, no cross-frame tracking), and CREPE-tiny (64 ms windows / 10 ms updates, no Viterbi) get new inference identities. Praat uses the shared range directly, without the app wrapper's padding or 40 Hz clamp; its silence reference is buffer-local. Attune running-p95, SwiftF0, SPICE and native Tartini must reuse compatible checkpoints. Their missing or incompatible evidence blocks the run **before any inference**. Neural models retain their fixed inference ranges and common final output range gate.

The dry-run saves exact per-track misses and reasons. Execution repeats preflight, resumes completed method/track checkpoints, and returns the new accuracy and **P95 feedback latency (ms)** table. This is the duration-weighted steady-state age of the latest available estimate, including held outputs, analysis look-ahead and simulated process-CPU computation/queueing. Startup and device/rendering delays are excluded; playback ends at audio duration without a drain period. Timing coverage and startup diagnostics, per-track geometry, derived metric caches and provenance are saved alongside the report. Paired tests below are recomputed from the returned rows. This is a configuration comparison on the already-used set, not held-out validation.


In [13]:
# Coverage preview; execution refreshes only Attune.
native_feedback_plan = BENCH.run_native_feedback_evaluation(dry_run=True)
display(native_feedback_plan.groupby(['dataset', 'model']).size().rename('retained_tracks'))


dataset          model         
bach10-original  attune             36
                 crepe_tiny         36
                 mcleod             36
                 praat              36
                 pyin_framewise     36
                 spice              36
                 swiftf0            36
coco             attune            260
                 crepe_tiny        260
                 mcleod            260
                 praat             260
                 pyin_framewise    260
                 spice             260
                 swiftf0           260
urmp             attune            136
                 crepe_tiny        136
                 mcleod            136
                 praat             136
                 pyin_framewise    136
                 spice             136
                 swiftf0           136
Name: retained_tracks, dtype: int64

In [14]:
# Refresh Attune online checkpoints and combine with frozen competitor results.
paper_online, paper_online_table = BENCH.run_native_feedback_evaluation(dry_run=False)
paper_online_paired = BENCH.show_paired_analysis(
    *paper_online,
    mode="Streaming",
    methods=PAPER_ONLINE_METHODS,
)
paper_online_table


Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[  33/36] attune: 32/36 finished | 0s | 32 cached; loading model  attune: 4 worker(s); forced 8-worker policy (limited only by remaining tracks)
[  36/36] attune: 35/36 finished | 34s | finished: 02-AchLiebenChristen-saxphone
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[ 110/136] attune: 109/136 finished | 0s | 109 cached; loading model  attune: 8 worker(s); forced 8-worker policy (limited only by remaining tracks)
[ 136/136] attune: 135/136 finished | 365s | finished: 42_Arioso_tpt_tpt_hn_tbn_tba/AuSep_3_hn_42_Arioso     
Streaming replay: 8 worker(s); serial frames per track; dedicated-worker CPU latency simulation
[ 260/260] attune: 260/260 finished | 0s | 260 cached


,recordings,output_updates,pooled_p95_output_latency_ms
model,,,
attune,432,7560617,47.1189
crepe_tiny,432,2195908,34.7938
mcleod,432,946560,23.3350
praat,432,6352577,15.4571
pyin_framewise,432,945264,47.1349
spice,432,686388,35.9980
swiftf0,432,1365615,160.0843


Streaming: frame-pooled scores on the common completed recording set; Holm correction within this mode. Confidence intervals are unadjusted.


,tracks,source_pieces,frames,voiced_frames,unvoiced_frames
dataset,,,,,
bach10-original,36,10,118942,105652,13290
coco,260,256,556720,527274,29446
urmp,136,27,1522704,1161066,361638


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm
model,,,,,
attune,0.9488,0.9576,0.9337,0.9747,0.1329
pyin_framewise,0.9597,0.9682,0.8236,0.9994,0.7802
praat,0.9539,0.9639,0.9151,0.9940,0.2573
swiftf0,0.8655,0.8685,0.8565,0.8867,0.1832
spice,0.8860,0.8978,0.8569,0.9407,0.2722
mcleod,0.9185,0.9303,0.8948,0.9489,0.2103
crepe_tiny,0.9253,0.9388,0.9160,0.9554,0.1249


,method,metric,tracks,source_pieces,excluded_tracks,eligible_frames,seed,bootstrap_resamples,difference_pp,ci_low_pp,ci_high_pp,p_value,permutations,p_holm,significant
0,pyin_framewise,Overall Accuracy,432,293,0,2198366,0,9999,11.0115,8.5675,12.9511,0.0001,9999,0.0018,True
1,pyin_framewise,Voicing Recall,432,293,0,1793992,0,9999,-2.4765,-3.3453,-1.8642,0.0001,9999,0.0018,True
2,pyin_framewise,Voicing False Alarm,432,293,0,404374,0,9999,-64.7299,-70.7782,-59.8059,0.0001,9999,0.0018,True
3,praat,Overall Accuracy,432,293,0,2198366,0,9999,1.8697,1.3964,2.4132,0.0001,9999,0.0018,True
4,praat,Voicing Recall,432,293,0,1793992,0,9999,-1.9348,-2.7109,-1.3960,0.0001,9999,0.0018,True
5,praat,Voicing False Alarm,432,293,0,404374,0,9999,-12.4437,-16.8678,-9.1412,0.0001,9999,0.0018,True
6,swiftf0,Overall Accuracy,432,293,0,2198366,0,9999,7.7221,6.2674,9.0201,0.0001,9999,0.0018,True
7,swiftf0,Voicing Recall,432,293,0,1793992,0,9999,8.7969,6.6886,10.6153,0.0001,9999,0.0018,True
8,swiftf0,Voicing False Alarm,432,293,0,404374,0,9999,-5.0288,-9.9912,-1.1540,0.0004,9999,0.0018,True
9,spice,Overall Accuracy,432,293,0,2198366,0,9999,7.6840,6.5775,8.8838,0.0001,9999,0.0018,True


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/pitch/notebook_runs/paired_analysis/streaming/23ff13a0e836


,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,recordings,timing_coverage_failures,evaluated_seconds,evaluated_duration_coverage,P95 feedback latency (ms)
model,,,,,,,,,,
attune,0.9488,0.9576,0.9337,0.9747,0.1329,432,0,21943.7762,0.9982,49.6090
crepe_tiny,0.9253,0.9388,0.9160,0.9554,0.1249,432,0,21955.1314,0.9987,43.9966
mcleod,0.9185,0.9303,0.8948,0.9489,0.2103,432,0,21974.0429,0.9995,45.3726
praat,0.9539,0.9639,0.9151,0.9940,0.2573,432,0,21976.8669,0.9997,27.1776
pyin_framewise,0.9597,0.9682,0.8236,0.9994,0.7802,432,0,21943.7295,0.9982,69.0384
spice,0.8860,0.8978,0.8569,0.9407,0.2722,432,0,21954.7458,0.9987,66.0962
swiftf0,0.8655,0.8685,0.8565,0.8867,0.1832,432,0,21842.3738,0.9936,175.1775


## Interpreting raw pitch accuracy

Raw pitch accuracy uses a 50-cent tolerance; raw chroma accuracy additionally ignores octave errors. Under these adapters' zero-frequency unvoiced convention, raw pitch accuracy decreases both when played tones are suppressed and when voiced estimates have the wrong pitch. The following decomposition uses pooled reference-voiced frame counts. It is not a mean of track-level percentages. Negative-frequency latent predictions, if introduced by another adapter, would require revisiting this decomposition.


In [15]:
rpa_decomposition = BENCH.show_rpa_decomposition(
    preliminary, preliminary_urmp, preliminary_coco, methods=PRIMARY_METHODS
)


Tracks  Raw Pitch Accuracy  \
Suite                             Method                               
pitch/preliminary_bach10_original pyin         8              0.9709   
                                  attune       8              0.9596   
                                  praat        8              0.9579   
                                  spice        8              0.8972   
pitch/preliminary_urmp            pyin        24              0.9591   
                                  attune      24              0.9460   
                                  praat       24              0.9314   
                                  spice       24              0.8921   
pitch/preliminary_coco            pyin        26              0.9648   
                                  attune      26              0.9612   
                                  praat       26              0.9474   
                                  spice       26              0.9352   

                                          Voicing Recall  \
Suite                             Method                   
pitch/preliminary_bach10_original pyin            0.9799   
                                  attune          0.9671   
                                  praat           0.9787   
                                  spice           0.9320   
pitch/preliminary_urmp            pyin            0.9867   
                                  attune          0.9701   
                                  praat           0.9571   
                                  spice           0.9470   
pitch/preliminary_coco            pyin            0.9868   
                                  attune          0.9798   
                                  praat           0.9836   
                                  spice           0.9722   

                                          Missed-voicing share  \
Suite                             Method                         
pitch/preliminary_bach10_original pyin                  0.0201   
                                  attune                0.0329   
                                  praat                 0.0213   
                                  spice                 0.0680   
pitch/preliminary_urmp            pyin                  0.0133   
                                  attune                0.0299   
                                  praat                 0.0429   
                                  spice                 0.0530   
pitch/preliminary_coco            pyin                  0.0132   
                                  attune                0.0202   
                                  praat                 0.0164   
                                  spice                 0.0278   

                                          Wrong-pitch share  \
Suite                             Method                      
pitch/preliminary_bach10_original pyin               0.0090   
                                  attune             0.0075   
                                  praat              0.0209   
                                  spice              0.0348   
pitch/preliminary_urmp            pyin               0.0276   
                                  attune             0.0240   
                                  praat              0.0257   
                                  spice              0.0548   
pitch/preliminary_coco            pyin               0.0219   
                                  attune             0.0186   
                                  praat              0.0363   
                                  spice              0.0369   

                                          RPA / Voicing Recall  
Suite                             Method                        
pitch/preliminary_bach10_original pyin                  0.9908  
                                  attune                0.9922  
                                  praat                 0.9787  
                                  spice                 0.9

## Full comparisons

These cells drop the preliminary per-instrument sampler and run every track in the configured corpora, including all 40 original Bach10 isolated recordings and all 149 isolated real-recording URMP stems. They are long and cache-backed: jobs run in up to `CONFIG.workers` spawned workers with one numerical-library thread per worker, except CREPE, which shares that CPU budget between processes and TensorFlow inference threads, and any method whose dependency is missing is skipped rather than failing the run.

**Skipped by Run All by default.** Set `RUN_FULL_RUNS = True` in the configuration cell to enable these suites.
The full audio suite contains original Bach10 plus the two MDB synthesized corpora. Its new `all_audio_original_bach10` result directory preserves the previous `all_audio` results.


In [16]:
if RUN_FULL_RUNS:
    all_audio = BENCH.run_all_audio()
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [17]:
if RUN_FULL_RUNS:
    all_urmp = BENCH.run_all_urmp(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


In [18]:
if RUN_FULL_RUNS:
    all_coco = BENCH.run_all_coco(force=False)
else:
    print("Skipped full run (RUN_FULL_RUNS=False).")


Skipped full run (RUN_FULL_RUNS=False).


## Leaderboard

One row per (method, suite), ranked by overall accuracy. Read `Voicing False Alarm` alongside it: the synthetic suites have effectively digital silence in their unvoiced frames, while original Bach10 and URMP supply real isolated recordings. Neither substitutes for a live microphone benchmark in a real room.

With `RUN_FULL_RUNS = False`, this table uses the three preliminary suites.

In [19]:
leaderboard = BENCH.show_leaderboard(
    *(
        (all_audio, all_urmp, all_coco)
        if RUN_FULL_RUNS
        else (preliminary, preliminary_urmp, preliminary_coco)
    )
)


,,Tracks,Raw Pitch Accuracy,Raw Chroma Accuracy,Overall Accuracy,Voicing Recall,Voicing False Alarm,Audio(s)/Compute(s)
model,suite,,,,,,,
attune,pitch/preliminary_coco,26,0.9612,0.9636,0.9557,0.9798,0.1434,7.8474
crepe,pitch/preliminary_coco,26,0.9617,0.9633,0.9507,0.9832,0.2489,0.1825
pyin,pitch/preliminary_coco,26,0.9648,0.9673,0.9500,0.9868,0.3191,6.1414
basic_pitch,pitch/preliminary_coco,26,0.9658,0.9663,0.9462,0.9959,0.4086,37.5741
praat,pitch/preliminary_coco,26,0.9474,0.9575,0.9423,0.9836,0.1491,679.2098
attune,pitch/preliminary_urmp,24,0.9460,0.9523,0.9339,0.9701,0.1063,8.2753
praat,pitch/preliminary_bach10_original,8,0.9579,0.9629,0.9312,0.9787,0.2763,273.7342
crepe,pitch/preliminary_bach10_original,8,0.9717,0.9723,0.9302,0.9809,0.3930,0.1887
attune,pitch/preliminary_bach10_original,8,0.9596,0.9602,0.9287,0.9671,0.3121,8.2137


## Terminal equivalents

The same suites from a shell, for long overnight runs. `--write` publishes clean-suite frame-pooled metrics to `results/pitch/pitch_frame_weighted_benchmarks.csv`. Per-track results are written under `results/pitch/raw_outputs/<method>/`.

```shell
# balanced preliminary Bach10 draw, plus dependency availability
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets bach10-original --per-instrument 2 --seed 0 --list

# full original Bach10 plus MDB synthesized corpora
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets mdb-stem-synth mdb-melody-synth bach10-original --write

# balanced preliminary URMP draw, then the complete 149-stem corpus
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets urmp --per-instrument 2 --seed 0
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets urmp --write

# balanced preliminary CocoChorales draw
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets coco --split test --per-instrument 2 --seed 0 --materialize --write

# the heavy PyTorch models in their own invocation, so runtimes do not stack
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py \
    --datasets bach10-original --methods torchcrepe penn --write

# resume an interrupted run, or re-score existing caches without detecting
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --skip-cached --write
at-venv/bin/python benchmarks/modules/pitch/PitchBenchmarker.py --datasets coco --cache-only --write
```